**Table 1: Model architectures and epistasis benchmark performance.** Nineteen genomic foundation models spanning three architectural families (supervised track prediction, masked language modeling, state-space/hybrid) evaluated across four epistasis benchmarks. eQTL z: distance-controlled within-bin permutation z-score comparing 1,451 statistically identified epistatic eQTL pairs against 27,343 LD-correlated controls (N_BINS = 1000, 500 permutations). FAS ρ: partial Spearman correlation between model epistasis metric and experimentally measured epistasis in FAS exon 6 saturation mutagenesis (16,728 pairs), controlling for genomic distance; best of raw $R_\text{singles}$ and Mahalanobis-calibrated distance reported per model. MST1R ρ: Spearman correlation between model epistasis and allele-exchange exon skipping percentage (966 pairs); best of raw and Mahalanobis. KRAS %ile: percentile rank of the known compensatory G60G–Q61K pair's magnitude ratio among 13,455 synthetic SNV×SNV pairs in the KRAS neighborhood (lower = stronger detection). Bold indicates best value per benchmark column. Models sorted by eQTL z-score. SpliceAI included as a splicing-specific baseline (FAS analysis only). Context lengths range from 256 nt (MutBERT) to 1M nt (HyenaDNA); embedding dimensions from 256 to 4,096; parameter counts from 6.6M (HyenaDNA) to 7.0B (Evo2).

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configuration via paper_data_config ──
sys.path.insert(0, str(Path("..").resolve()))
from paper_data_config import EPISTASIS_PAPER_ROOT

OUT_DIR = EPISTASIS_PAPER_ROOT / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# Model metadata — from published papers and model cards
# ═══════════════════════════════════════════════════════════════════════

models = [
    # Track-based
    {"key": "borzoi", "display": "Borzoi", "category": "Track",
     "architecture": "Dilated CNN + Transformer", "params_M": 250,
     "emb_dim": 1920, "max_input_nt": 524_288, "tokenization": "1-hot",
     "training": "Supervised (expression/epigenomic tracks)",
     "reference": "Linder et al. 2023"},
     
    {"key": "alphagenome", "display": "AlphaGenome", "category": "Track",
     "architecture": "CNN + Transformer", "params_M": 400,
     "emb_dim": 1536, "max_input_nt": 262_144, "tokenization": "1-hot",
     "training": "Supervised (expression/splicing tracks)",
     "reference": "DeepMind 2024"},
     
    {"key": "spliceai", "display": "SpliceAI", "category": "Track",
     "architecture": "Dilated CNN", "params_M": 19,
     "emb_dim": 256, "max_input_nt": 10_000, "tokenization": "1-hot",
     "training": "Supervised (splice sites)",
     "reference": "Jaganathan et al. 2019"},

    # Nucleotide Transformer family
    {"key": "nt50_multi", "display": "NT-50M", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 50,
     "emb_dim": 512, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt50_3mer", "display": "NT-50M-3mer", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 50,
     "emb_dim": 512, "max_input_nt": 6_000, "tokenization": "3-mer",
     "training": "MLM (multi-species, 3-mer)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt100_multi", "display": "NT-100M", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 100,
     "emb_dim": 640, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt250_multi", "display": "NT-250M", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 250,
     "emb_dim": 1024, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt500_multi", "display": "NT-500M", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 500,
     "emb_dim": 1024, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt500_ref", "display": "NT-500M-ref", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 500,
     "emb_dim": 1024, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (human reference only)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt2500_multi", "display": "NT-2.5B", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 2500,
     "emb_dim": 2560, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species)",
     "reference": "Dalla-Torre et al. 2023"},
     
    {"key": "nt2500_okgp", "display": "NT-2.5B-1kGP", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 2500,
     "emb_dim": 2560, "max_input_nt": 6_000, "tokenization": "6-mer",
     "training": "MLM (multi-species + 1kGP fine-tune)",
     "reference": "Dalla-Torre et al. 2023"},

    # Other MLMs
    {"key": "dnabert", "display": "DNABERT", "category": "MLM",
     "architecture": "BERT encoder", "params_M": 117,
     "emb_dim": 768, "max_input_nt": 512, "tokenization": "BPE",
     "training": "MLM (human genome)",
     "reference": "Zhou et al. 2024"},
     
    {"key": "mutbert", "display": "MutBERT", "category": "MLM",
     "architecture": "BERT encoder", "params_M": 86,
     "emb_dim": 768, "max_input_nt": 256, "tokenization": "k-mer",
     "training": "MLM (mutation-aware)",
     "reference": "Yang et al. 2024"},
     
    {"key": "rinalmo", "display": "RiNALMo", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 650,
     "emb_dim": 1280, "max_input_nt": 1022, "tokenization": "1-mer (RNA)",
     "training": "MLM (RNA sequences)",
     "reference": "Penic et al. 2024"},
     
    {"key": "specieslm", "display": "SpeciesLM", "category": "MLM",
     "architecture": "Transformer encoder", "params_M": 270,
     "emb_dim": 1024, "max_input_nt": 512, "tokenization": "BPE",
     "training": "MLM (species-conditioned)",
     "reference": "Zwirn & Linial 2024"},

    # State-space / hybrid
    {"key": "hyenadna", "display": "HyenaDNA", "category": "SSM",
     "architecture": "Hyena (long conv)", "params_M": 6.6,
     "emb_dim": 256, "max_input_nt": 1_000_000, "tokenization": "char-level",
     "training": "Autoregressive (human genome)",
     "reference": "Nguyen et al. 2024"},
     
    {"key": "caduceus", "display": "Caduceus", "category": "SSM",
     "architecture": "Mamba (bidirectional)", "params_M": 16,
     "emb_dim": 256, "max_input_nt": 131_072, "tokenization": "char-level",
     "training": "MLM (RC-equivariant)",
     "reference": "Schiff et al. 2024"},
     
    {"key": "evo2", "display": "Evo2", "category": "Hybrid",
     "architecture": "StripedHyena (SSM+Attention)", "params_M": 7000,
     "emb_dim": 4096, "max_input_nt": 131_072, "tokenization": "char-level",
     "training": "Autoregressive (multi-species)",
     "reference": "Brixi et al. 2025"},
     
    {"key": "convnova", "display": "ConvNova", "category": "CNN",
     "architecture": "Dilated CNN", "params_M": 7,
     "emb_dim": 640, "max_input_nt": 2048, "tokenization": "1-hot",
     "training": "Contrastive (species pairs)",
     "reference": "Lynn & Tuller 2024"},
]

model_df = pd.DataFrame(models)

# Format params nicely
def fmt_params(m):
    if m >= 1000:
        return f"{m/1000:.1f}B"
    return f"{m}M"

model_df["params"] = model_df["params_M"].apply(fmt_params)

# Format max input
def fmt_input(n):
    if n >= 1_000_000:
        return f"{n/1_000_000:.0f}M"
    if n >= 1000:
        return f"{n/1000:.0f}k"
    return str(n)

model_df["max_input"] = model_df["max_input_nt"].apply(fmt_input)

print("Model metadata table:")
display_cols = ["display", "category", "architecture", "params", "emb_dim", "max_input", "tokenization", "training"]
print(model_df[display_cols].to_string(index=False))
print(f"\n{len(model_df)} models total")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Benchmark performance — from notebook outputs
# ═══════════════════════════════════════════════════════════════════════

# eQTL z-scores vs topLD correlated (N_BINS=1000)
eqtl_z_topld = {
    "Borzoi": 55.9, "AlphaGenome": 19.3, "Evo2": 6.3, "NT-500M": 7.9,
    "NT-100M": 3.5, "NT-2.5B": 1.0, "RiNALMo": 0.4, "ConvNova": -2.2,
    "HyenaDNA": -4.7, "NT-250M": -0.7, "Caduceus": -2.7, "MutBERT": -1.3,
    "NT-50M-3mer": -3.4, "NT-50M": -3.2, "NT-500M-ref": -5.2,
    "SpeciesLM": -4.0, "NT-2.5B-1kGP": -5.2, "DNABERT": -3.6,
}

# eQTL z-scores vs GTEx distance-matched null (N_BINS=1000)
eqtl_z_gtex = {
    "Borzoi": 8.9, "AlphaGenome": 2.9, "Evo2": 3.5, "NT-500M": -2.7,
    "NT-100M": -2.2, "NT-2.5B": -1.6, "RiNALMo": 0.0, "ConvNova": -4.6,
    "HyenaDNA": -0.7, "NT-250M": -4.8, "Caduceus": 0.5, "MutBERT": -3.1,
    "NT-50M-3mer": -3.9, "NT-50M": -2.6, "NT-500M-ref": -4.1,
    "SpeciesLM": -4.0, "NT-2.5B-1kGP": -4.8, "DNABERT": -4.9,
}

# FAS partial Spearman rho (best of epi_R vs Mahalanobis)
fas_rho = {
    "AlphaGenome": 0.068, "DNABERT": 0.060, "HyenaDNA": 0.058,
    "MutBERT": 0.056, "NT-500M-ref": 0.055, "RiNALMo": 0.050,
    "Borzoi": 0.052, "NT-100M": 0.041, "SpeciesLM": 0.040,
    "NT-250M": 0.039, "Caduceus": 0.036, "NT-2.5B-1kGP": 0.036,
    "NT-500M": 0.035, "ConvNova": 0.033, "NT-2.5B": 0.032,
    "NT-50M": 0.029, "NT-50M-3mer": 0.028, "Evo2": 0.026,
}

# MST1R Spearman rho with exon skipping (best of epi_R vs Mahalanobis)
mst1r_rho = {
    "Borzoi": 0.096, "AlphaGenome": 0.075, "NT-500M-ref": 0.065,
    "SpeciesLM": 0.051, "ConvNova": 0.047, "HyenaDNA": 0.046,
    "NT-50M-3mer": 0.040, "NT-2.5B-1kGP": 0.037, "Caduceus": 0.029,
    "RiNALMo": 0.018, "NT-250M": -0.004, "DNABERT": -0.007,
    "NT-2.5B": -0.010, "NT-50M": -0.011, "NT-100M": -0.022,
    "NT-500M": -0.027, "MutBERT": -0.044,
}

# KRAS G60G-Q61K percentile
kras_pctile = {
    "AlphaGenome": 0.04, "Borzoi": 1.3, "NT-500M": 1.4,
    "NT-250M": 1.4, "DNABERT": 2.0, "NT-2.5B": 5.6,
    "NT-500M-ref": 6.5, "SpeciesLM": 8.6, "NT-50M": 10.1,
    "NT-100M": 16.0, "Caduceus": 23.9, "RiNALMo": 24.0,
    "HyenaDNA": 41.0, "MutBERT": 45.8, "NT-2.5B-1kGP": 54.2,
    "ConvNova": 64.2, "NT-50M-3mer": 65.8,
}

# Merge into model_df
for col_name, data_dict in [("eQTL_z_topld", eqtl_z_topld), ("eQTL_z_gtex", eqtl_z_gtex),
                              ("FAS_rho", fas_rho), ("MST1R_rho", mst1r_rho),
                              ("KRAS_pctile", kras_pctile)]:
    model_df[col_name] = model_df["display"].map(data_dict)

# ── Format final table ──
final_cols = ["display", "category", "params", "emb_dim", "max_input",
              "eQTL_z_topld", "eQTL_z_gtex", "FAS_rho", "MST1R_rho", "KRAS_pctile"]

table = model_df[final_cols].copy()
table.columns = ["Model", "Type", "Params", "Emb dim", "Max input",
                  "eQTL z (LD)", "eQTL z (GTEx)", "FAS ρ", "MST1R ρ", "KRAS %ile"]

table = table.sort_values("eQTL z (LD)", ascending=False, na_position="last")

print(table.to_string(index=False, float_format=lambda x: f"{x:.2f}" if abs(x) < 100 else f"{x:.1f}"))

table.to_csv(OUT_DIR / "model_summary_table.tsv", sep="\t", index=False)
print(f"\nSaved to {OUT_DIR / 'model_summary_table.tsv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Render publication-quality table (booktabs style, tight)
# ═══════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

mm = 1 / 25.4

tbl = model_df.copy()
tbl = tbl.sort_values("eQTL_z_topld", ascending=False, na_position="last")

def fmt_z(v):
    if pd.isna(v): return "—"
    return f"{v:.1f}"

def fmt_rho(v):
    if pd.isna(v): return "—"
    return f"{v:.3f}"

def fmt_kras(v):
    if pd.isna(v): return "—"
    if v < 1: return f"{v:.2f}"
    return f"{v:.1f}"

def fmt_params(v):
    if v >= 1000: return f"{v/1000:.1f}B"
    if v >= 10: return f"{v:.0f}M"
    return f"{v:.1f}M"

def fmt_input(v):
    if v >= 1_000_000: return f"{v/1_000_000:.0f}M"
    if v >= 1000: return f"{v/1000:.0f}k"
    return str(int(v))

rows = []
for _, r in tbl.iterrows():
    rows.append([
        r['display'], r['category'], fmt_params(r['params_M']),
        str(int(r['emb_dim'])), fmt_input(r['max_input_nt']),
        fmt_z(r.get('eQTL_z_topld')), fmt_z(r.get('eQTL_z_gtex')),
        fmt_rho(r.get('FAS_rho')), fmt_rho(r.get('MST1R_rho')),
        fmt_kras(r.get('KRAS_pctile')),
    ])

col_labels = ["Model", "Type", "Params", "Dim", "Input",
              "eQTL z\n(LD)", "eQTL z\n(GTEx)", "FAS ρ", "MST1R ρ", "KRAS\n%ile"]

best_idx = {}
for j, col in enumerate(col_labels):
    if "eQTL" in col or "FAS" in col or "MST1R" in col:
        vals = [float(r[j]) for r in rows if r[j] != "—"]
        if vals:
            best_val = max(vals)
            best_idx[j] = [i for i, r in enumerate(rows) if r[j] != "—" and float(r[j]) == best_val]
    elif "KRAS" in col:
        vals = [float(r[j]) for r in rows if r[j] != "—"]
        if vals:
            best_val = min(vals)
            best_idx[j] = [i for i, r in enumerate(rows) if r[j] != "—" and float(r[j]) == best_val]

n_rows = len(rows)
row_h = 0.20
fig_h = row_h * (n_rows + 2.8)
fig_w = 183 * mm

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, n_rows + 2.8)

col_x =  [0.00, 0.14, 0.21, 0.28, 0.35, 0.44, 0.53, 0.63, 0.76, 0.89]
col_ha = ["left", "center", "center", "center", "center",
          "center", "center", "center", "center", "center"]

y_top = n_rows + 2.3
ax.plot([0, 1], [y_top, y_top], color="black", linewidth=1.0, clip_on=False)

y_header = n_rows + 1.7
for j, label in enumerate(col_labels):
    ax.text(col_x[j], y_header, label, ha=col_ha[j], va="center",
            fontsize=5.5, fontweight="bold", fontfamily="serif", color="black")

y_hrule = n_rows + 1.15
ax.plot([0, 1], [y_hrule, y_hrule], color="black", linewidth=0.6, clip_on=False)

for i, row in enumerate(rows):
    y = n_rows - i * 1.0 + 0.5
    for j, val in enumerate(row):
        is_best = j in best_idx and i in best_idx[j]
        weight = "bold" if is_best else "normal"
        ax.text(col_x[j], y, val, ha=col_ha[j], va="center",
                fontsize=5, fontweight=weight, fontfamily="serif", color="black")

y_bot = 0.2
ax.plot([0, 1], [y_bot, y_bot], color="black", linewidth=1.0, clip_on=False)

caption = ("eQTL z (LD): vs TopLD-correlated controls. "
           "eQTL z (GTEx): vs GTEx distance-matched nulls. "
           "FAS ρ: partial Spearman (best of raw/Mahal). "
           "MST1R ρ: Spearman with exon skipping. "
           "KRAS %ile: G60G–Q61K MR percentile. Bold = best.")
ax.text(0.0, -0.05, caption, fontsize=4, color="#444444", va="top",
        fontfamily="serif", wrap=True)

for ext in (".png", ".pdf"):
    fig.savefig(OUT_DIR / f"table_models{ext}", dpi=600,
                bbox_inches="tight", facecolor="white", edgecolor="none",
                pad_inches=0.03)
plt.show()
print(f"Saved table_models to {OUT_DIR}")